In [ ]:
#from cellpose import models
#import pyclesperanto as cle
# from bioio import BioImage
# from skimage.segmentation import relabel_sequential
# from skimage.morphology import remove_small_objects
# from skimage.transform import rescale, resize
# from skimage.io import imsave, imread
# from skimage.measure import label, regionprops, regionprops_table, marching_cubes, mesh_surface_area
# from scipy import ndimage
# import numpy as np
# from glob import glob
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import scipy as sc
#from tqdm import tqdm
#from joblib import Parallel, delayed

#cle.select_device("NVIDIA")

In [ ]:
def read_imgs(path):
    files = sorted(glob(f"{path}/*.czi"))
    names = list(map(os.path.basename,files))
    names = [name[0:-4] for name in names]
    img_data = [BioImage(f).get_image_data("ZYX", C=1) for f in files]
    pixel_spacing = get_pixel_spacing(BioImage(files[0]))
    return img_data, pixel_spacing, names

def get_pixel_spacing(img):
    ''' 
    Use one image from dataset read in with BioImage without .get_image_data to access pixel spacing
    '''
    if img.dims['Z'][0] > 1:
        pixel_spacing = (img.physical_pixel_sizes[0],img.physical_pixel_sizes[1],img.physical_pixel_sizes[2])
    else:
        pixel_spacing = (img.physical_pixel_sizes[0],img.physical_pixel_sizes[1])
    return pixel_spacing

def normalize_images(input_image,tophat_radius=10):
    input_gpu = cle.push(input_image)
    #normalizing the image stack
    equalized_intensities_stack = cle.create_like(input_gpu)
    a_slice = cle.create([input_gpu.shape[1], input_gpu.shape[2]])
    num_slices = input_gpu.shape[0]
    mean_intensity_stack = cle.mean_of_all_pixels(input_gpu)
    corrected_slice = None
    for z in range(0, num_slices):
        # get a single slice out of the stack
        cle.copy_slice(input_gpu, a_slice, z)
        # measure its intensity
        mean_intensity_slice = cle.mean_of_all_pixels(a_slice)
        # correct the intensity
        correction_factor = mean_intensity_slice/mean_intensity_stack
        corrected_slice = cle.multiply_image_and_scalar(a_slice, corrected_slice, correction_factor)
        # copy slice back in a stack
        cle.copy_slice(corrected_slice, equalized_intensities_stack, z)
    #background subtraction (increase the signal to noise ratio for improved segmentation results)
    background_subtracted_top_hat = cle.top_hat_sphere(equalized_intensities_stack,radius_x=tophat_radius,radius_y=tophat_radius,radius_z=tophat_radius)
    #pull data off gpu
    input_pull = cle.pull(input_gpu)
    background_subtracted_top_hat_pull = cle.pull(background_subtracted_top_hat)
    equalized_intensities_stack_pull = cle.pull(equalized_intensities_stack)
    return background_subtracted_top_hat_pull

def down_size(img, scale=None):
   scale == 1 if scale == None else scale == scale
   if scale == None:
       print('no scaling provided, using scaling of 1 (no scaling applied)')
   return rescale(img, scale, anti_aliasing=True, channel_axis=None,preserve_range=True)

def run_cpsam(imgs):
    '''
    Give function list of all resized images for more efficient processing
    '''
    #io.logger_setup()
    model = models.CellposeModel(gpu=True)
    masks, _, _ = model.eval(imgs, batch_size=32, z_axis=0, do_3D=False,normalize=True,stitch_threshold=0.6,cellprob_threshold=-0.5)
    return masks

def up_size_masks(masks, img):
    if masks.shape == img.shape:
        print('masks and image are the same size, no resizing applied')
        return masks
    else:
        return resize(masks,img.shape,preserve_range=True,order=0).astype(np.uint16)

def clear_xy_borders(mask_img,border):
    bool_arr = np.zeros((mask_img.shape[0],mask_img.shape[1]-border*2,mask_img.shape[2]-border*2))
    xy_boarder = np.pad(bool_arr,pad_width=((0,0),(border,border),(border,border)),mode='constant',constant_values=1).astype(np.bool)
    labels, number = label(mask_img,background=0,return_num=True)
    indices = np.arange(number+1)
    border_indices = np.unique(labels[xy_boarder])
    label_mask = np.isin(indices, border_indices)
    mask = label_mask[labels.reshape(-1)].reshape(labels.shape)
    labels[mask] = 0
    return labels
     

def process_masks(mask_img, border, save_path, name):
    cleared_borders = clear_xy_borders(mask_img, border)
    rmv_small_objs = remove_small_objects(cleared_borders,max_size=200)
    seq_lables, _, _ = relabel_sequential(rmv_small_objs)
    imsave(os.path.join(save_path,f'{name}_cell_masks.tif'),seq_lables,check_contrast=False)
    return seq_lables

def _label_sa_solidity_pre(label_val, submask, pixel_spacing):
    m = submask.astype(np.uint8)

    # surface area
    padded = np.pad(m, 1, mode='constant')
    verts, faces, _, _ = marching_cubes(padded, level=0.5, spacing=pixel_spacing)
    surface_area = mesh_surface_area(verts, faces)

    # solidity on the same small crop — convex hull is cheap here
    sub_props = regionprops(m)[0]
    solidity = sub_props.solidity

    return label_val, surface_area, solidity


def measure_objects(mask_img, pixel_spacing, name, n_jobs=15):
    # 1. Standard descriptors — one fast pass.
    #    Pass spacing so 'area' is volume in physical units (e.g. µm³).
    #print('collecting regions properties')
    props_df = pd.DataFrame(regionprops_table(
        mask_img,
        spacing=pixel_spacing,
        properties=[
            'label', 'area', 'centroid', 'equivalent_diameter_area', 
            'extent', 'axis_major_length', 'axis_minor_length'
        ],
    ))

    # 2. Pre-crop each object in the main process so workers receive only
    #    a tiny bool array, not the full label volume.
    #print('finding objects with ndimage')
    objects = ndimage.find_objects(mask_img)
    #print(f'extracting subregions for {len(objects)}')
    tasks = []
    for idx, sl in enumerate(objects):
        if sl is None:
            continue
        label_val = idx + 1
        padded_sl = tuple(slice(max(0, s.start - 1), s.stop + 1) for s in sl)
        submask = (mask_img[padded_sl] == label_val)   # small bool array
        tasks.append((label_val, submask))

    # 3. Parallelize only the costly surface-area extraction.
    #print('Extracting suface areas for objects')
    results = Parallel(n_jobs=n_jobs)(
        delayed(_label_sa_solidity_pre)(label_val, submask, pixel_spacing)
        for label_val, submask in tasks
    )
    sa_df = pd.DataFrame(results, columns=['label', 'surface_area', 'solidity'])
    # 4. Merge on label — order-independent, gap-safe.
    #print(props_df.columns)
    df = props_df.merge(sa_df, on='label', how='left')
    # 5. Sphericity as a vectorized op, now that V and A are aligned columns.
    #    'area' is volume because spacing was passed to regionprops.
    df['sphericity'] = (np.pi**(1/3) * (6 * df['area'])**(2/3)) / df['surface_area']
    df['Image'] = name
    return df

def assign_condition(r):
    if 'WT_ctrl' in r['Image']:
        return 'WT-ctrl'
    elif 'DKO_ctrl' in r['Image']:
        return 'DKO-ctrl'
    else:
        return 'Unknown'

def merge_dfs_save(dfs, save_loc):
    all_data = pd.concat(dfs,ignore_index=True)
    all_data['group'] = all_data.apply(lambda x: assign_condition(x), axis=0)
    all_data.to_csv(os.path.join(save_loc,'Zebrafish_desmosome_df.tsv'), sep='\t',index=False)

    return all_data

In [ ]:
# full run
path = r"E:\KristinG\20260625_zebrafish\fin"
save_img_path = r"E:\KristinG\20260625_zebrafish\fin_outputs\Cell_Masks"
save_df_path = r"E:\KristinG\20260625_zebrafish\fin_outputs"
scale = [0.5,0.2,0.2]
print('reading in image data')
imgs, pixel_spacing, names = read_imgs(path)
print('normalizing images')
norm_imgs = [normalize_images(i) for i  in tqdm(imgs)]
del(imgs)
print('rescaling')
rescaled_imgs = [down_size(i,scale) for i in tqdm(norm_imgs)]
print('running cpsam')
masks = tqdm(run_cpsam(rescaled_imgs))
del(rescaled_imgs)
print('resizing masks')
resized_masks = [up_size_masks(m, i) for m, i in tqdm(zip(masks,norm_imgs))]
del(norm_imgs)
del(masks)
print('clearing objects from XY border and saving')
cleared_borders = [clear_xy_borders(m, border = 6, name = n, save_path = save_img_path) for m,n in tqdm(zip(resized_masks, names))]
del(resized_masks)
print('getting object measurements')
dfs = [measure_objects(m,pixel_spacing,n) for m, n in tqdm(zip(cleared_borders,names))]
print('saving data frame')
merge_dfs_save(dfs,save_df_path)
del(cleared_borders,dfs)

In [ ]:
masks_path = r"E:\KristinG\20260625_zebrafish\fin_outputs\Cell_Masks"
save_new_masks_path = r"E:\KristinG\20260625_zebrafish\fin_outputs\Filtered_Cell_Masks"
save_df_path = r"E:\KristinG\20260625_zebrafish\fin_outputs"
pixel_spacing = get_pixel_spacing(BioImage(r"E:\KristinG\20260625_zebrafish\fin\DKO_ctrl_1_skin_1_ventral_fin_Out.czi"))
mask_files = sorted(glob(rf'{masks_path}\*.tif'))
masks = list(map(imread,mask_files))

# Plotting Data

In [ ]:
import pandas.plotting as pdplt

In [ ]:
batch1 = pd.read_csv('/home/gkristin/Zebrafish_desmosomes/Zebrafish_desmosome_tail_data.tsv',sep='\t')
batch2 = pd.read_csv('/home/gkristin/Zebrafish_desmosomes/Zebrafish_desmosome_df_2nd_batch.tsv',sep='\t')

In [ ]:
batch1['batch'] = 1

In [ ]:
batch1.rename(columns={'groups':'group','regions':'region'},inplace=True)

In [ ]:
batch2.columns

In [ ]:
merged_df = pd.concat([batch1,batch2],ignore_index=True)

In [ ]:
merged_df.to_csv('/home/gkristin/Zebrafish_desmosomes/Zebrafish_desmosome_df_tail_all_batches.tsv',sep='\t',index=False)

In [ ]:
fish_median = merged_df.groupby(['batch','group','fish_id'],as_index=False).median(numeric_only=True)

In [ ]:
fish_median.tail()

In [ ]:
batch1_DKO = fish_median.query('batch == 1 & group == "DKO-ctrl"')
batch2_DKO = fish_median.query('batch == 2 & group == "DKO-ctrl"')
batch1_WT = fish_median.query('batch == 1 & group == "WT-ctrl"')
batch2_WT = fish_median.query('batch == 2 & group == "WT-ctrl"')

wt = fish_median.query('group == "WT-ctrl"')
dko = fish_median.query('group == "DKO-ctrl"')


In [ ]:
batch1_WT['surface_area'].to_numpy()

In [ ]:
#testing batch differences:
wt_sa_batches = sc.stats.ttest_ind(batch1_WT['surface_area'].to_numpy(),batch2_WT['surface_area'].to_numpy(),equal_var=False)
wt_sp_batches = sc.stats.ttest_ind(batch1_WT['sphericity'].to_numpy(),batch2_WT['sphericity'].to_numpy(),equal_var=False)
dko_sa_batches = sc.stats.ttest_ind(batch1_DKO['surface_area'].to_numpy(),batch2_DKO['surface_area'].to_numpy(),equal_var=False)
dko_sp_batches = sc.stats.ttest_ind(batch1_DKO['sphericity'].to_numpy(),batch2_DKO['sphericity'].to_numpy(),equal_var=False)


In [ ]:
print(f"""
    p-values from the following tests:
    WT batches S.A. Welch's T-test: {wt_sa_batches.pvalue}
    WT batches Sphericity Welch's T-test: {wt_sp_batches.pvalue}
    DKO batches S.A. Welch's T-test: {dko_sa_batches.pvalue}
    DKO batches Sphericity Welch's T-test: {dko_sp_batches.pvalue}
"""
    )

In [ ]:
wt_v_dko_sa = sc.stats.ttest_ind(wt['surface_area'].to_numpy(),dko['surface_area'].to_numpy(),equal_var=False)
st_v_dko_sphericity = sc.stats.ttest_ind(wt['sphericity'].to_numpy(),dko['sphericity'].to_numpy(),equal_var=False)
print(f"""
    p-values from the following tests:
    WT v DKO S.A. Welch's T-test: {wt_v_dko_sa.pvalue}
    WT v DKO Sphericity Welch's T-test: {st_v_dko_sphericity.pvalue}
"""
    )

In [ ]:
# create plots for easy vis
fig, axs = plt.subplots(2,3,figsize=(10,8))
sns.violinplot(data=fish_median, x='group', y='area', hue='batch', split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[0,0])
axs[0,0].set_title('Vol')
axs[0,0].set_ylabel('Vol (um³)')
sns.violinplot(data=fish_median, x='group', y='equivalent_diameter_area',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[0,1])
axs[0,1].set_title('Equivalent Diameter Vol')
axs[0,1].set_ylabel('Vol (um³)')
sns.violinplot(data=fish_median, x='group', y='extent',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[0,2])
axs[0,2].set_title('Extent')
axs[0,2].set_ylabel('Extent [0-1]')
sns.violinplot(data=fish_median, x='group', y='surface_area',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[1,0])
axs[1,0].set_title('Surface Area')
axs[1,0].set_ylabel('Area (um²)')
sns.violinplot(data=fish_median, x='group', y='solidity',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[1,1])
axs[1,1].set_title('Solidity')
axs[1,1].set_ylabel('Solidity [0-1]')
sns.violinplot(data=fish_median, x='group', y='sphericity',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[1,2])
axs[1,2].set_title('Sphericity')
axs[1,2].set_ylabel('Sphericity [0-1]')
plt.tight_layout()
plt.savefig('/home/gkristin/Zebrafish_desmosomes/Cell_Shape_Features_Plot_Batches_fish_medians.png',dpi=300)
plt.show()

In [ ]:
pg = sns.PairGrid(fish_median,hue='batch',vars=['area','equivalent_diameter_area', 'extent', 'surface_area', 'solidity', 'sphericity'])
pg.map_upper(sns.scatterplot,style=fish_median['group'])
pg.map_diag(sns.histplot,)
pg.add_legend()
#pg.savefig('/home/gkristin/Zebrafish_desmosomes/Cell_Shape_Features_Plot_Batches_fish_medians.png',dpi=300)

Plots to make:
Use the Vol, equivalent, SA, and Sphericity just for tail as individual plots
Some relationship plotting of the above measurements

In [ ]:
df = pd.read_table('/home/gkristin/Zebrafish_desmosomes/Zebrafish_desmosome_df_added_filter_columns.tsv',sep='\t')

In [ ]:
g = sns.catplot(data=df_tail, x='groups', y='equivalent_diameter_area', hue='groups',kind='violin',fill=False, order=['WT-ctrl','DKO-ctrl'],inner='quart')
g.set_axis_labels('Groups','Vol (um³)')
g.set_titles(col_template='Volume per cell')

In [ ]:
fig, axs = plt.subplots(1,1,figsize=(5,5))
sns.violinplot(data=df_tail, x='groups', y='sphericity', hue='groups', fill=False, order=['WT-ctrl','DKO-ctrl'],inner='quart',ax=axs)
axs.set_title('Sphericity per Cell')
axs.set_ylabel('Sphericity [0-1]')
axs.set_ylim(0,1)
plt.tight_layout()
plt.savefig('/home/gkristin/Zebrafish_desmosomes/Sphericity_Plot.png',dpi=300)

In [ ]:
from itertools import combinations

In [ ]:
grouped_df = (df.groupby('Image').median(numeric_only=True))

In [ ]:
def assign_condition(r):
    if 'WT' in r:
        return 'WT'
    elif 'DKO' in r:
        return 'DKO'
grouped_df['groups'] = [assign_condition(r) for r in grouped_df.index]

In [ ]:
def assign_condition(r):
    if 'tail' in r:
        return 'tail'
    elif 'dorsal' in r:
        return 'dorsal'
    elif 'ventral' in r:
        return 'ventral'
grouped_df['regions'] = [assign_condition(r) for r in grouped_df.index]

In [ ]:
measurements = ['area','equivalent_diameter_area','surface_area','sphericity']

In [ ]:
combos = list(combinations(measurements,2))

In [ ]:
pg = sns.PairGrid(fish_median.query("group == 'DKO-ctrl'"),hue='batch',diag_sharey=False,vars=measurements,palette=sns.color_palette(palette='colorblind',n_colors=2))
pg.map_lower(sns.scatterplot)
pg.map_diag(sns.histplot,bins=10)
pg.map_upper(sns.regplot,scatter=True,logx=True)
pg.add_legend(title='Batch ID',adjust_subtitles=True)
pg.savefig('/home/gkristin/Zebrafish_desmosomes/Paired_Grid_Plot_DKO_Batches_Fish_Median.png',dpi=300)


In [ ]:
sns.lmplot(df.query("regions == 'tail'"),x=combos[1][0],y=combos[0][1],hue='groups')

In [ ]:
df_tail = grouped_df.query("regions == 'tail'")

In [ ]:
df_tail.value_counts('groups')

In [ ]:
def assign_fish_id(r):
    values = r.split('_')
    fish_id = values[2]
    return fish_id
grouped_df['fish_id'] = [assign_fish_id(r) for r in grouped_df.index]


In [ ]:
grouped_df

In [ ]:
df_tail = grouped_df.query("regions == 'tail'")

In [ ]:
df_tail.value_counts(['groups'])